In [1]:
import pandas as pd

# CARGAR LOS 4 ARCHIVOS
q1_2019 = pd.read_csv("Divvy_Trips_2019_Q1.csv")
q2_2019 = pd.read_csv("Divvy_Trips_2019_Q2.csv")
q3_2019 = pd.read_csv("Divvy_Trips_2019_Q3.csv")
q4_2019 = pd.read_csv("Divvy_Trips_2019_Q4.csv")

# RENOMBRAR COLUMNAS DE Q2 (nombres raros) para que coincidan con Q1/Q3/Q4
q2_2019 = q2_2019.rename(columns={
    '01 - Rental Details Rental ID': 'trip_id',
    '01 - Rental Details Local Start Time': 'start_time',
    '01 - Rental Details Local End Time': 'end_time',
    '01 - Rental Details Bike ID': 'bikeid',
    '01 - Rental Details Duration In Seconds Uncapped': 'tripduration',
    '03 - Rental Start Station ID': 'from_station_id',
    '03 - Rental Start Station Name': 'from_station_name',
    '02 - Rental End Station ID': 'to_station_id',
    '02 - Rental End Station Name': 'to_station_name',
    'User Type': 'usertype',
    'Member Gender': 'gender',
    '05 - Member Details Member Birthday Year': 'birthyear'
})

# VERIFICAR que ahora Q2 tiene las mismas columnas que Q1
print("Columnas Q1:", q1_2019.columns.tolist())
print("Columnas Q2 tras renombrar:", q2_2019.columns.tolist())

# RENOMBRAR TODOS al formato final (igual que el script original)
def rename_to_standard(df):
    return df.rename(columns={
        'trip_id': 'ride_id',
        'bikeid': 'rideable_type',
        'start_time': 'started_at',
        'end_time': 'ended_at',
        'from_station_name': 'start_station_name',
        'from_station_id': 'start_station_id',
        'to_station_name': 'end_station_name',
        'to_station_id': 'end_station_id',
        'usertype': 'member_casual'
    })

q1_2019 = rename_to_standard(q1_2019)
q2_2019 = rename_to_standard(q2_2019)
q3_2019 = rename_to_standard(q3_2019)
q4_2019 = rename_to_standard(q4_2019)

# CONVERTIR ride_id y rideable_type a string
for df in [q1_2019, q2_2019, q3_2019, q4_2019]:
    df['ride_id'] = df['ride_id'].astype(str)
    df['rideable_type'] = df['rideable_type'].astype(str)

# UNIR TODO EN UN SOLO DATAFRAME
all_trips = pd.concat([q1_2019, q2_2019, q3_2019, q4_2019], ignore_index=True)

print("\nTotal de filas:", len(all_trips))
print("Columnas:", all_trips.columns.tolist())

# ELIMINAR columnas que no usaremos
all_trips = all_trips.drop(columns=['birthyear', 'gender', 'tripduration'], errors='ignore')

# LIMPIAR member_casual
print("\nValores antes de limpiar:", all_trips['member_casual'].value_counts())
all_trips['member_casual'] = all_trips['member_casual'].replace({
    'Subscriber': 'member',
    'Customer': 'casual'
})
print("Valores después de limpiar:", all_trips['member_casual'].value_counts())

# CONVERTIR FECHAS
all_trips['started_at'] = pd.to_datetime(all_trips['started_at'])
all_trips['ended_at'] = pd.to_datetime(all_trips['ended_at'])

# AÑADIR COLUMNAS DE FECHA
all_trips['date'] = all_trips['started_at'].dt.date
all_trips['month'] = all_trips['started_at'].dt.month
all_trips['day'] = all_trips['started_at'].dt.day
all_trips['year'] = all_trips['started_at'].dt.year
all_trips['day_of_week'] = all_trips['started_at'].dt.day_name()

# CALCULAR ride_length en segundos
all_trips['ride_length'] = (all_trips['ended_at'] - all_trips['started_at']).dt.total_seconds()

# ELIMINAR DATOS MALOS
all_trips_v2 = all_trips[
    (all_trips['start_station_name'] != "HQ QR") &
    (all_trips['ride_length'] >= 0)
].copy()

print("\nFilas tras limpieza:", len(all_trips_v2))

# ANÁLISIS DESCRIPTIVO
print("\nEstadísticas de ride_length:\n", all_trips_v2['ride_length'].describe())

print("\nPor tipo de usuario:\n",
      all_trips_v2.groupby('member_casual')['ride_length'].agg(['mean', 'median', 'max', 'min']))

# ORDENAR DÍAS
days_order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
all_trips_v2['day_of_week'] = pd.Categorical(all_trips_v2['day_of_week'], categories=days_order, ordered=True)

# RESUMEN FINAL
summary_stats = all_trips_v2.groupby(['member_casual', 'day_of_week']).agg(
    number_of_rides=('ride_id', 'count'),
    average_duration=('ride_length', 'mean')
).reset_index()

print("\nResumen por tipo y día:\n", summary_stats)

# EXPORTAR
summary_stats.to_csv('avg_ride_length.csv', index=False)
print("\nArchivo exportado correctamente.")

Columnas Q1: ['trip_id', 'start_time', 'end_time', 'bikeid', 'tripduration', 'from_station_id', 'from_station_name', 'to_station_id', 'to_station_name', 'usertype', 'gender', 'birthyear']
Columnas Q2 tras renombrar: ['trip_id', 'start_time', 'end_time', 'bikeid', 'tripduration', 'from_station_id', 'from_station_name', 'to_station_id', 'to_station_name', 'usertype', 'gender', 'birthyear']

Total de filas: 3818004
Columnas: ['ride_id', 'started_at', 'ended_at', 'rideable_type', 'tripduration', 'start_station_id', 'start_station_name', 'end_station_id', 'end_station_name', 'member_casual', 'gender', 'birthyear']

Valores antes de limpiar: member_casual
Subscriber    2937367
Customer       880637
Name: count, dtype: int64
Valores después de limpiar: member_casual
member    2937367
casual     880637
Name: count, dtype: int64

Filas tras limpieza: 3817991

Estadísticas de ride_length:
 count    3.817991e+06
mean     1.450466e+03
std      2.985231e+04
min      6.100000e+01
25%      4.110000e+

/tmp/ipykernel_15958/3487414905.py:103: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_stats = all_trips_v2.groupby(['member_casual', 'day_of_week']).agg(



Resumen por tipo y día:
    member_casual day_of_week  number_of_rides  average_duration
0         casual      Sunday           170173       3371.111692
1         casual      Monday           101489       3269.993359
2         casual     Tuesday            88655       3444.797033
3         casual   Wednesday            89745       3620.043980
4         casual    Thursday           101372       3597.067356
5         casual      Friday           121141       3610.536639
6         casual    Saturday           208056       3243.666623
7         member      Sunday           256234        924.174110
8         member      Monday           458780        854.957032
9         member     Tuesday           497025        849.155519
10        member   Wednesday           494277        828.590685
11        member    Thursday           486915        826.787511
12        member      Friday           456966        833.848656
13        member    Saturday           287163        978.162305

Archivo expor